# Market Making Fundamentals

Audience: Python learners preparing for quant interviews. Prerequisites: basic Python and pandas.

Goals: Bid/ask, RFQs, side conventions, inventory and cash.

Outline: setup, short tasks, checks, pitfalls. Run cells from top to bottom. All market data here is synthetic.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from market_simulator import (MarketMaker, ClientModel, MarketEnvironment,
    QuotedTrade, CompletedTrade, generate_market_data, generate_rfqs, run_simulation, make_quote)
from market_simulator.risk import beta, historical_volatility

# Resolve synthetic data from the installed local package, independent of kernel CWD.
import market_simulator
DATA = Path(market_simulator.__file__).resolve().parents[1] / 'data'
prices = pd.read_csv(DATA / 'market_prices.csv', parse_dates=['timestamp'])
requests = pd.read_csv(DATA / 'client_rfqs.csv', parse_dates=['timestamp'])
print(prices.head(2).to_string(index=False))


## Two-sided price

Given a positive mid and full spread fraction, return (bid, ask). Example 100 and 0.02 -> (99,101). Reject invalid inputs.


In [ ]:
def two_sided(mid: float, spread: float) -> tuple[float, float]:
    if not np.isfinite(mid) or mid <= 0 or not np.isfinite(spread) or not 0 < spread < 2:
        raise ValueError('invalid mid or spread')
    return (mid * (1 - spread/2), mid * (1 + spread/2))

**Reasoning and complexity:** O(1) time and space. The spread fraction here is full spread; 2% on each side would require 0.04.

**Common mistake:** Forgetting the stated side, time ordering, or validation contract. An explicit loop is often a useful alternative when debugging.

In [ ]:
assert two_sided(100, .02) == (99, 101)

## Join RFQs to prices

Return an RFQ DataFrame enriched with `mid`, matching timestamp and ticker. Preserve all RFQs. Raise ValueError if any match is missing or duplicated.


In [ ]:
def enrich_rfqs(rfqs: pd.DataFrame, market: pd.DataFrame) -> pd.DataFrame:
    if market.duplicated(['timestamp', 'ticker']).any():
        raise ValueError('duplicate price key')
    result = rfqs.merge(market[['timestamp','ticker','mid']], on=['timestamp','ticker'], how='left', validate='many_to_one')
    if result['mid'].isna().any():
        raise ValueError('missing price')
    return result

**Reasoning and complexity:** Hash join is typically O(n+m) expected time and O(n+m) space. A nested scan is much slower.

**Common mistake:** Forgetting the stated side, time ordering, or validation contract. An explicit loop is often a useful alternative when debugging.

In [ ]:
joined = enrich_rfqs(requests, prices)
assert len(joined) == len(requests) and joined.mid.notna().all()

## Maker cash update

Return (new_inventory, new_cash). A maker buy of q at p adds q shares and spends qp; a maker sell subtracts q shares and receives qp. Fee rate is applied to notional on either side.


In [ ]:
def apply_fill(inventory: int, cash: float, side: str, q: int, p: float, fee_rate: float = 0.0) -> tuple[int, float]:
    if side not in {'buy','sell'} or type(q) is not int or q <= 0 or p <= 0 or not 0 <= fee_rate < 1:
        raise ValueError('invalid fill')
    delta = q if side == 'buy' else -q
    return inventory + delta, cash - delta*p - q*p*fee_rate

**Reasoning and complexity:** O(1) time and space. Do not confuse client direction with maker direction.

**Common mistake:** Forgetting the stated side, time ordering, or validation contract. An explicit loop is often a useful alternative when debugging.

In [ ]:
assert apply_fill(0, 1000, 'buy', 2, 100) == (2, 800)
assert apply_fill(0, 1000, 'sell', 2, 100) == (-2, 1200)

## Extension
Try a different random seed or add a transaction fee. Explain how your answer changes.